In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast

movies = pd.read_csv('../data/processed/movies_cleaned.csv')

# these columns were saved as strings, convert back to lists
movies['genres'] = movies['genres'].apply(ast.literal_eval)
movies['keywords'] = movies['keywords'].apply(ast.literal_eval)
movies['top_cast'] = movies['top_cast'].apply(ast.literal_eval)

print("Loaded:", movies.shape)
movies.head(2)

Loaded: (4800, 11)


,id,title,genres,keywords,overview,vote_average,vote_count,release_date,runtime,top_cast,director
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...",7.2,11800,2009-12-10,162.0,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha...",6.9,4500,2007-05-19,169.0,"[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski


In [4]:
def clean_name(name):
    # remove spaces from multi-word names so they become one token
    # "Science Fiction" → "sciencefiction", "James Cameron" → "jamescameron"
    return name.replace(" ", "").lower()

def build_soup(row):
    genres = [clean_name(g) for g in row['genres']]
    keywords = [clean_name(k) for k in row['keywords']]
    cast = [clean_name(c) for c in row['top_cast']]
    director = clean_name(row['director'])
    overview = row['overview'].lower()
    
    # director appears twice — gives it more weight than cast
    return ' '.join(genres) + ' ' + \
           ' '.join(keywords) + ' ' + \
           ' '.join(cast) + ' ' + \
           director + ' ' + director + ' ' + \
           overview

movies['soup'] = movies.apply(build_soup, axis=1)

# verify
print("Example soup for Avatar:")
print(movies['soup'][0][:300])

Example soup for Avatar:
action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron jamescameron in the 


In [6]:
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)

# fit and transform — this is where the magic happens
tfidf_matrix = tfidf.fit_transform(movies['soup'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Each movie is now a vector of", tfidf_matrix.shape[1], "numbers")

TF-IDF matrix shape: (4800, 10000)
Each movie is now a vector of 10000 numbers


In [8]:
# compute pairwise cosine similarity between all movies
# this gives us a 4800x4800 matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine similarity matrix shape:", cosine_sim.shape)
print("Similarity between Avatar and movie 1:", cosine_sim[0][1])

Cosine similarity matrix shape: (4800, 4800)
Similarity between Avatar and movie 1: 0.015398089924812043


In [10]:
# create a reverse map — movie title → index in dataframe
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def get_content_recommendations(title, n=10):
    # get index of the movie
    if title not in indices:
        return f"Movie '{title}' not found in database"
    
    idx = indices[title]
    
    # get similarity scores for this movie with all others
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # sort by similarity score descending
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # skip index 0 (that's the movie itself, similarity = 1.0)
    sim_scores = sim_scores[1:n+1]
    
    # get movie indices
    movie_indices = [i[0] for i in sim_scores]
    scores = [round(i[1], 3) for i in sim_scores]
    
    # return results
    result = movies[['title', 'genres', 'vote_average']].iloc[movie_indices].copy()
    result['similarity_score'] = scores
    
    return result

# TEST IT
get_content_recommendations("Interstellar")

,title,genres,vote_average,similarity_score
1709,Space Pirate Captain Harlock,"[Animation, Science Fiction]",6.5,0.161
365,Contact,"[Drama, Science Fiction, Mystery]",7.2,0.158
635,Apollo 13,[Drama],7.3,0.152
270,The Martian,"[Drama, Adventure, Science Fiction]",7.6,0.144
4330,Silent Running,"[Adventure, Drama, Science Fiction]",6.3,0.143
643,Space Cowboys,"[Action, Adventure, Thriller]",6.3,0.137
96,Inception,"[Action, Thriller, Science Fiction, Mystery, A...",8.1,0.130
149,Armageddon,"[Action, Thriller, Science Fiction, Adventure]",6.4,0.130
1033,Insomnia,"[Crime, Mystery, Thriller]",6.8,0.125
1196,The Prestige,"[Drama, Mystery, Thriller]",8.0,0.125


In [12]:
print("=== The Dark Knight ===")
print(get_content_recommendations("The Dark Knight"))
print()
print("=== The Godfather ===")
print(get_content_recommendations("The Godfather"))
print()
print("=== Toy Story ===")
print(get_content_recommendations("Toy Story"))

=== The Dark Knight ===
                                        title  \
3                       The Dark Knight Rises   
119                             Batman Begins   
428                            Batman Returns   
3853  Batman: The Dark Knight Returns, Part 2   
299                            Batman Forever   
1359                                   Batman   
9          Batman v Superman: Dawn of Justice   
210                            Batman & Robin   
1196                             The Prestige   
1033                                 Insomnia   

                                genres  vote_average  similarity_score  
3     [Action, Crime, Drama, Thriller]           7.6             0.454  
119             [Action, Crime, Drama]           7.5             0.365  
428                  [Action, Fantasy]           6.6             0.333  
3853               [Action, Animation]           7.9             0.313  
299           [Action, Crime, Fantasy]           5.2             0.271 

In [16]:
# IMDB's weighted rating formula:
# WR = (v/(v+m)) * R + (m/(v+m)) * C
# v = vote count for this movie
# m = minimum votes required (we use 75th percentile)
# R = average rating for this movie
# C = mean rating across all movies

C = movies['vote_average'].mean()
m = movies['vote_count'].quantile(0.75)

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m)) * R + (m/(v+m)) * C

movies['weighted_score'] = movies.apply(weighted_rating, axis=1)

print(f"Mean rating C: {C:.2f}")
print(f"Minimum votes m: {m:.0f}")
print("\nTop 10 movies by weighted score:")
print(movies[['title', 'vote_average', 'vote_count', 'weighted_score']]
      .sort_values('weighted_score', ascending=False).head(10))

Mean rating C: 6.09
Minimum votes m: 737

Top 10 movies by weighted score:
                         title  vote_average  vote_count  weighted_score
1881  The Shawshank Redemption           8.5        8205        8.301546
3336             The Godfather           8.4        5893        8.143464
662                 Fight Club           8.3        9413        8.139691
3231              Pulp Fiction           8.3        8428        8.122463
65             The Dark Knight           8.2       12002        8.078058
809               Forrest Gump           8.2        7927        8.020706
96                   Inception           8.1       13752        7.997874
1818          Schindler's List           8.3        4329        7.978821
3864                  Whiplash           8.3        4254        7.973995
95                Interstellar           8.1       10867        7.972484


In [18]:
def get_content_recommendations(title, n=10):
    if title not in indices:
        return f"Movie '{title}' not found in database"
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+20]  # get extra candidates
    movie_indices = [i[0] for i in sim_scores]
    scores = [round(i[1], 3) for i in sim_scores]
    
    # build result dataframe
    result = movies[['title', 'genres', 'vote_average', 
                      'vote_count', 'weighted_score']].iloc[movie_indices].copy()
    result['similarity_score'] = scores
    
    # combined score — similarity matters more than popularity
    result['final_score'] = (0.7 * result['similarity_score'] / result['similarity_score'].max()) + \
                             (0.3 * result['weighted_score'] / result['weighted_score'].max())
    
    # sort by final score and return top n
    result = result.sort_values('final_score', ascending=False).head(n)
    
    return result[['title', 'genres', 'vote_average', 
                   'similarity_score', 'final_score']].reset_index(drop=True)

# test
print("=== Interstellar (improved) ===")
get_content_recommendations("Interstellar")

=== Interstellar (improved) ===


,title,genres,vote_average,similarity_score,final_score
0,Contact,"[Drama, Science Fiction, Mystery]",7.2,0.158,0.939527
1,Space Pirate Captain Harlock,"[Animation, Science Fiction]",6.5,0.161,0.931200
2,Apollo 13,[Drama],7.3,0.152,0.917828
3,The Martian,"[Drama, Adventure, Science Fiction]",7.6,0.144,0.903178
4,Inception,"[Action, Thriller, Science Fiction, Mystery, A...",8.1,0.130,0.862240
5,Silent Running,"[Adventure, Drama, Science Fiction]",6.3,0.143,0.849505
6,The Prestige,"[Drama, Mystery, Thriller]",8.0,0.125,0.830397
7,Space Cowboys,"[Action, Adventure, Thriller]",6.3,0.137,0.824647
8,Armageddon,"[Action, Thriller, Science Fiction, Adventure]",6.4,0.130,0.800287
9,Insomnia,"[Crime, Mystery, Thriller]",6.8,0.125,0.785745


In [20]:
import pickle

# save cosine similarity matrix
with open('../models/cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)

# save movies dataframe with soup and weighted score
movies.to_csv('../data/processed/movies_final.csv', index=False)

print("Model saved!")
print("cosine_sim.pkl:", cosine_sim.shape)

Model saved!
cosine_sim.pkl: (4800, 4800)


In [22]:
print("=== The Godfather (improved) ===")
print(get_content_recommendations("The Godfather"))
print()
print("=== Toy Story (improved) ===")
print(get_content_recommendations("Toy Story"))


=== The Godfather (improved) ===
                     title                             genres  vote_average  \
0   The Godfather: Part II                     [Drama, Crime]           8.3   
1  The Godfather: Part III           [Crime, Drama, Thriller]           7.1   
2          The Cotton Club     [Music, Drama, Crime, Romance]           6.6   
3    Peggy Sue Got Married  [Comedy, Drama, Fantasy, Romance]           5.9   
4           Apocalypse Now                       [Drama, War]           8.0   
5            The Rainmaker           [Drama, Crime, Thriller]           6.7   
6            The Outsiders                     [Crime, Drama]           6.9   
7         New York Stories           [Comedy, Drama, Romance]           6.2   
8   The Master of Disguise                   [Comedy, Family]           3.7   
9       This Thing of Ours          [Drama, Action, Thriller]           5.0   

   similarity_score  final_score  
0             0.406     1.000000  
1             0.316     0.8